<a href="https://colab.research.google.com/github/NicholasVunZhunMin/PL/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-generativeai

In [3]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [5]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make decisions or predictions.


In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [7]:
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        current_datetime = datetime.now().strftime('%Y-%m-%d %H:%M:%S') # 修改為包含時間的格式
        grades.append([current_datetime, subject, grade])
        print(f"已記錄：日期和時間: {current_datetime}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [8]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思，並提供學習建議。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要、常見迷思整理，以及針對這些成績的學習建議（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要與學習建議... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要和學習建議生成失敗。"

### 從 Google Sheet 讀取現有成績

以下函數用於從 Google Sheet 中讀取所有已儲存的成績資料，並將其顯示為 Pandas DataFrame。

In [9]:
def get_all_grades_from_sheet():
    """
    從 Google Sheet 讀取所有已儲存的成績資料。
    """
    global _auth_done, _gc, _ws

    if not _auth_done:
        print("--- 正在進行 Google Sheet 身份驗證... ---")
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True

    try:
        sh = _gc.open_by_url(SHEET_URL)
        _ws = sh.worksheet(WORKSHEET_NAME)

        # 讀取所有資料，跳過標題行
        data = _ws.get_all_values()
        if not data or len(data) < 1: # Original: len(data) < 2
            print("Google Sheet 中沒有可用的資料。")
            return pd.DataFrame(columns=REQUIRED_COLUMNS)

        # Use the first row as headers, assuming they are consistent
        headers = data[0]
        df = pd.DataFrame(data[1:], columns=headers)

        print("--- 已成功從 Google Sheet 讀取成績。---")
        return df
    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
        return pd.DataFrame()
    except Exception as e:
        print(f"讀取 Google Sheet 時發生未預期的錯誤：{e}")
        return pd.DataFrame()

### 示範：讀取並顯示現有成績

In [10]:
existing_grades_df = get_all_grades_from_sheet()
display(existing_grades_df)

--- 正在進行 Google Sheet 身份驗證... ---
--- 已成功從 Google Sheet 讀取成績。---


,Gemini 給的收支建議,科目,作業成績,,
0,您好！很高興看到您開始記錄收支，這是理財的第一步，也是最重要的一步！\n\n從您提供的這四筆...,,90,,
1,2025-09-25,網際網路概論,85,,
2,2025-09-25,程式語言作業一,95,,
3,2025-09-25,程式語言作業二,93,,
4,2025-09-25,網際網路概論作業一,90,,
...,...,...,...,...,...
61,,,2026-04-04,英文,100
62,,,2026-04-07,math,80
63,,,2026-04-07,english,60
64,,,2026-04-07,chinese,70


In [11]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---


KeyboardInterrupt: Interrupted by user

In [ ]:
new_grades

In [ ]:
get_ai_summary(new_grades)

In [ ]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

### 使用 Gradio 建立成績管理介面

我們將建立一個 Gradio 介面，讓使用者可以：
1. 輸入科目和成績。
2. 將成績新增到一個列表中。
3. 基於列表中的成績生成 AI 摘要。
4. 將新成績和 AI 摘要寫入 Google Sheet。

為了確保 Google Sheet 的連線在 Gradio 應用程式中能夠正確執行，我們將在 Gradio 介面啟動前進行一次性的認證。

In [ ]:
# 確保 Google Sheet 認證在 Gradio 應用啟動前完成
if not _auth_done:
    print("--- 正在進行 Google Sheet 身份驗證 (為 Gradio 介面準備)... ---")
    auth.authenticate_user()
    creds, _ = default()
    _gc = gspread.authorize(creds)
    sh = _gc.open_by_url(SHEET_URL)
    _ws = sh.worksheet(WORKSHEET_NAME)
    _auth_done = True
    print("--- Google Sheet 連線成功。---")
else:
    # 如果已經認證過，確保 _ws 也被設定
    if _ws is None:
        sh = _gc.open_by_url(SHEET_URL)
        _ws = sh.worksheet(WORKSHEET_NAME)


def add_grade_to_list(subject, grade, grades_list):
    """
    將新的科目和成績新增到成績列表中。
    """
    if not subject or not grade:
        return grades_list, "請輸入科目和成績！"

    try:
        grade_int = int(grade)
    except ValueError:
        return grades_list, "成績必須是數字！"

    current_datetime = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    grades_list.append([current_datetime, subject, grade_int])

    # 顯示為 DataFrame，並返回更新後的列表作為 Gradio State
    df_display = pd.DataFrame(grades_list, columns=REQUIRED_COLUMNS)
    return grades_list, f"已新增：{subject} - {grade_int}", df_display

def process_grades_and_summary(grades_list):
    """
    處理累積的成績，生成 AI 摘要，並將成績和摘要寫入 Google Sheet。
    """
    global _ws # 使用 global _ws 確保寫入正確的工作表

    if not grades_list:
        return "沒有輸入任何成績，無法生成摘要。", pd.DataFrame(columns=REQUIRED_COLUMNS)

    # 將新成績寫入 Google Sheet
    try:
        _ws.append_rows(grades_list)
        print("\n--- 成績已成功寫入 Google Sheet。---")
    except Exception as e:
        return f"寫入 Google Sheet 時發生錯誤：{e}", pd.DataFrame(columns=REQUIRED_COLUMNS)

    # 獲取 AI 摘要並寫入 Google Sheet
    summary = get_ai_summary(grades_list)

    try:
        # 尋找第一行空白列
        next_row = len(_ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        _ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        _ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            _ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
    except Exception as e:
        summary += f"\n\n警告：寫入 AI 摘要到 Google Sheet 時發生錯誤：{e}"

    return summary, pd.DataFrame(columns=REQUIRED_COLUMNS) # 清空顯示的成績列表

def clear_all(grades_list):
    """
    清除所有輸入和狀態。
    """
    return [], "", pd.DataFrame(columns=REQUIRED_COLUMNS)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def generate_grade_trend_plot(grades_df):
    """
    生成學生成績趨勢圖。
    """
    if grades_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, '沒有足夠的成績資料來繪製趨勢圖', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
        ax.axis('off')
        return fig

    # 確保資料型別正確
    grades_df['日期'] = pd.to_datetime(grades_df['日期'])
    grades_df['作業成績'] = pd.to_numeric(grades_df['作業成績'], errors='coerce')
    grades_df.dropna(subset=['作業成績'], inplace=True)

    if grades_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, '沒有足夠的有效成績資料來繪製趨勢圖', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
        ax.axis('off')
        return fig

    grades_df = grades_df.sort_values(by='日期')

    fig = plt.figure(figsize=(12, 6))
    sns.lineplot(data=grades_df, x='日期', y='作業成績', marker='o')
    plt.title('學生成績趨勢')
    plt.xlabel('日期')
    plt.ylabel('作業成績')
    plt.grid(True)
    plt.tight_layout()
    return fig

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 學生成績管理與 AI 學習建議系統")

    # Gradio State 用於儲存成績列表
    grades_state = gr.State([])

    with gr.Row():
        with gr.Column():
            gr.Markdown("## 輸入新成績")
            subject_input = gr.Textbox(label="科目", placeholder="例如：數學、英文")
            grade_input = gr.Number(label="成績", placeholder="請輸入數字成績")
            add_btn = gr.Button("新增成績")
            add_status = gr.Textbox(label="新增狀態", interactive=False)

        with gr.Column():
            gr.Markdown("## 當前待處理成績")
            current_grades_display = gr.DataFrame(headers=REQUIRED_COLUMNS, label="已新增成績")

    with gr.Row():
        process_btn = gr.Button("生成 AI 摘要並儲存所有成績")
        clear_btn = gr.Button("清除所有")

    gr.Markdown("## AI 學習建議")
    summary_output = gr.Textbox(label="AI 摘要和建議", interactive=False, lines=10)

    # 事件處理
    add_btn.click(
        add_grade_to_list,
        inputs=[subject_input, grade_input, grades_state],
        outputs=[grades_state, add_status, current_grades_display]
    )

    process_btn.click(
        process_grades_and_summary,
        inputs=[grades_state],
        outputs=[summary_output, current_grades_display] # 清空顯示的成績列表
    )

    clear_btn.click(
        clear_all,
        inputs=[grades_state],
        outputs=[grades_state, add_status, current_grades_display]
    )

demo.launch(debug=True, share=True)